# Healthcare Analytics: Data Cleaning, EDA & Billing Prediction
**Author:** Karan Pratap

**Dataset:** Healthcare Dataset (Kaggle, `prasad22/healthcare-dataset`) - file `healthcare_dataset.csv`

**Workflow:** Data Cleaning -> Exploratory Data Analysis -> Machine Learning (Regression + Classification) -> Recommendations

> Note: Is dataset ko `healthcare_dataset.csv` naam se is notebook ke same folder me rakhna hai.

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, classification_report, confusion_matrix)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="viridis")
os.makedirs("figures", exist_ok=True)
RESULTS = {}

def save_fig(name):
    plt.tight_layout()
    plt.savefig(f"figures/{name}.png", dpi=150)
    plt.show()

## 1. Data Loading & Overview

In [ ]:
df = pd.read_csv("healthcare_dataset.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()
print("\nMissing values per column:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.describe()

## 2. Data Cleaning
Steps: (1) column names standardize, (2) duplicates hatana, (3) Name ki inconsistent capitalization fix karna,
(4) dates ko datetime me convert karna, (5) negative / invalid billing amounts handle karna, (6) new features banana.

In [ ]:
raw_rows = len(df)
df.columns = [c.strip().replace(" ", "_") for c in df.columns]

# 1. Duplicates
dups = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

# 2. Text standardization (names come with mixed upper/lower case)
df["Name"] = df["Name"].str.strip().str.title()
for col in ["Doctor", "Hospital"]:
    df[col] = df[col].astype(str).str.strip()

# 3. Dates
df["Date_of_Admission"] = pd.to_datetime(df["Date_of_Admission"])
df["Discharge_Date"] = pd.to_datetime(df["Discharge_Date"])

# 4. Invalid billing (negative values)
neg_bill = int((df["Billing_Amount"] < 0).sum())
df = df[df["Billing_Amount"] > 0].reset_index(drop=True)

# 5. Feature engineering
df["Length_of_Stay"] = (df["Discharge_Date"] - df["Date_of_Admission"]).dt.days
df["Admission_Month"] = df["Date_of_Admission"].dt.month
df["Admission_Year"] = df["Date_of_Admission"].dt.year
df["Age_Group"] = pd.cut(df["Age"], bins=[0, 18, 35, 50, 65, 120],
                         labels=["0-18", "19-35", "36-50", "51-65", "65+"])

RESULTS["cleaning"] = {"raw_rows": raw_rows, "duplicates_removed": dups,
                       "negative_billing_removed": neg_bill, "clean_rows": len(df)}
print(RESULTS["cleaning"])
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(df["Age"], bins=30, kde=True, ax=ax[0]); ax[0].set_title("Patient Age Distribution")
sns.histplot(df["Billing_Amount"], bins=40, kde=True, ax=ax[1]); ax[1].set_title("Billing Amount Distribution")
save_fig("01_age_billing_distribution")

In [ ]:
plt.figure(figsize=(9, 4.5))
order = df["Medical_Condition"].value_counts().index
sns.countplot(data=df, y="Medical_Condition", order=order)
plt.title("Patients by Medical Condition")
save_fig("02_condition_counts")

In [ ]:
plt.figure(figsize=(9, 4.5))
avg = df.groupby("Medical_Condition")["Billing_Amount"].mean().sort_values()
avg.plot(kind="barh"); plt.title("Average Billing Amount by Medical Condition"); plt.xlabel("Average Billing (USD)")
save_fig("03_avg_billing_condition")
RESULTS["top_billing_condition"] = [avg.idxmax(), float(avg.max())]
RESULTS["low_billing_condition"] = [avg.idxmin(), float(avg.min())]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.countplot(data=df, x="Admission_Type", ax=ax[0]); ax[0].set_title("Admission Type")
sns.countplot(data=df, x="Test_Results", ax=ax[1]); ax[1].set_title("Test Results")
save_fig("04_admission_test_results")

In [ ]:
plt.figure(figsize=(9, 4.5))
ins = df.groupby("Insurance_Provider")["Billing_Amount"].mean().sort_values()
ins.plot(kind="bar"); plt.title("Average Billing by Insurance Provider"); plt.ylabel("Average Billing (USD)")
plt.xticks(rotation=0)
save_fig("05_billing_insurance")

In [ ]:
monthly = df.groupby(df["Date_of_Admission"].dt.to_period("M")).size()
monthly.index = monthly.index.to_timestamp()
plt.figure(figsize=(11, 4.5))
monthly.plot(); plt.title("Monthly Admissions Trend"); plt.ylabel("Admissions")
save_fig("06_monthly_admissions")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(df["Length_of_Stay"], bins=30, ax=ax[0]); ax[0].set_title("Length of Stay (days)")
sns.boxplot(data=df, x="Age_Group", y="Billing_Amount", ax=ax[1]); ax[1].set_title("Billing by Age Group")
save_fig("07_los_agegroup")

In [ ]:
plt.figure(figsize=(6, 5))
corr = df[["Age", "Billing_Amount", "Length_of_Stay", "Admission_Month", "Room_Number"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
save_fig("08_correlation")
RESULTS["billing_los_corr"] = float(corr.loc["Billing_Amount", "Length_of_Stay"])
RESULTS["billing_age_corr"] = float(corr.loc["Billing_Amount", "Age"])

In [ ]:
RESULTS["eda"] = {
    "mean_age": float(df["Age"].mean()),
    "mean_billing": float(df["Billing_Amount"].mean()),
    "mean_los": float(df["Length_of_Stay"].mean()),
    "most_common_condition": df["Medical_Condition"].mode()[0],
    "most_common_admission": df["Admission_Type"].mode()[0],
    "date_from": str(df["Date_of_Admission"].min().date()),
    "date_to": str(df["Date_of_Admission"].max().date()),
}
RESULTS["eda"]

## 4. Machine Learning
### 4.1 Regression: Billing Amount predict karna
Models: Linear Regression, Decision Tree, Random Forest. Features: Age, Gender, Blood Type, Medical Condition,
Admission Type, Insurance Provider, Medication, Test Results, Length of Stay, Admission Month.
Name / Doctor / Hospital ko high-cardinality hone ki wajah se chhod diya gaya hai.

In [ ]:
num_cols = ["Age", "Length_of_Stay", "Admission_Month"]
cat_cols = ["Gender", "Blood_Type", "Medical_Condition", "Admission_Type",
            "Insurance_Provider", "Medication", "Test_Results"]
X = df[num_cols + cat_cols]
y = df["Billing_Amount"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def make_pre(scale=False):
    num = StandardScaler() if scale else "passthrough"
    return ColumnTransformer([("num", num, num_cols),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])

models = {
    "Linear Regression": Pipeline([("pre", make_pre(True)), ("m", LinearRegression())]),
    "Decision Tree": Pipeline([("pre", make_pre()), ("m", DecisionTreeRegressor(max_depth=6, min_samples_leaf=50, random_state=42))]),
    "Random Forest": Pipeline([("pre", make_pre()), ("m", RandomForestRegressor(n_estimators=150, max_depth=10, min_samples_leaf=20, n_jobs=-1, random_state=42))]),
}
rows, preds = [], {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    p = pipe.predict(X_test); preds[name] = p
    rows.append({"Model": name, "MAE": mean_absolute_error(y_test, p),
                 "RMSE": float(np.sqrt(mean_squared_error(y_test, p))), "R2": r2_score(y_test, p)})
reg_results = pd.DataFrame(rows).round(3)
RESULTS["regression"] = reg_results.to_dict(orient="records")
RESULTS["baseline_mae"] = float(mean_absolute_error(y_test, np.full(len(y_test), y_train.mean())))
reg_results

In [ ]:
best = reg_results.sort_values("R2", ascending=False).iloc[0]["Model"]
RESULTS["best_regression_model"] = best
plt.figure(figsize=(6, 5))
plt.scatter(y_test, preds[best], alpha=0.25, s=10)
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, "r--"); plt.xlabel("Actual Billing"); plt.ylabel("Predicted Billing")
plt.title(f"Actual vs Predicted ({best})")
save_fig("09_actual_vs_predicted")

In [ ]:
rf = models["Random Forest"]
names = rf.named_steps["pre"].get_feature_names_out()
imp = pd.Series(rf.named_steps["m"].feature_importances_, index=names).sort_values(ascending=False).head(10)
plt.figure(figsize=(8, 4.5)); imp[::-1].plot(kind="barh")
plt.title("Top 10 Feature Importances (Random Forest)")
save_fig("10_feature_importance")
RESULTS["top_features"] = {k: float(v) for k, v in imp.head(5).items()}
imp.head(5)

### 4.2 Classification: Test Results predict karna
Random Forest classifier se `Test_Results` (Normal / Abnormal / Inconclusive) predict karne ki koshish.

In [ ]:
cnum = ["Age", "Length_of_Stay", "Admission_Month", "Billing_Amount"]
ccat = ["Gender", "Blood_Type", "Medical_Condition", "Admission_Type", "Insurance_Provider", "Medication"]
Xc, yc = df[cnum + ccat], df["Test_Results"]
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)
clf = Pipeline([("pre", ColumnTransformer([("num", "passthrough", cnum),
                                           ("cat", OneHotEncoder(handle_unknown="ignore"), ccat)])),
                ("m", RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=20, n_jobs=-1, random_state=42))])
clf.fit(Xc_tr, yc_tr)
pc = clf.predict(Xc_te)
acc = accuracy_score(yc_te, pc)
RESULTS["classification"] = {"accuracy": float(acc), "chance_level": float(1 / yc.nunique())}
print("Accuracy:", round(acc, 3), "| Chance level:", round(1 / yc.nunique(), 3))
print(classification_report(yc_te, pc))
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(confusion_matrix(yc_te, pc, labels=clf.classes_), annot=True, fmt="d", cmap="Blues",
            xticklabels=clf.classes_, yticklabels=clf.classes_)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix - Test Results")
save_fig("11_confusion_matrix")

## 5. Key Findings & Recommendations
Neeche wali cell aapke actual results se findings print karti hai. Inhe report me use karo.

In [ ]:
r = RESULTS
print("Rows after cleaning:", r["cleaning"]["clean_rows"], "(from", r["cleaning"]["raw_rows"], ")")
print("Highest avg billing condition:", r["top_billing_condition"])
print("Lowest avg billing condition :", r["low_billing_condition"])
print("Best regression model:", r["best_regression_model"])
print("Classification accuracy:", round(r["classification"]["accuracy"], 3),
      "vs chance", round(r["classification"]["chance_level"], 3))

**Interpretation tips (apne results ke hisaab se edit karo):**
- Agar R2 0 ke aas-paas aur accuracy chance level (~33%) ke paas hai, to iska matlab ye Kaggle dataset synthetic hai aur features
  aur target ke beech real relationship nahi hai. Ye ek valid finding hai: report me likho ki model ne koi strong pattern nahi seekha
  aur real hospital data me clinical features (diagnosis codes, procedures, lab values) add karne se performance improve ho sakti hai.
- Cleaning ke steps (duplicates, negative billing, name formatting) ko report me zaroor highlight karo, wahi is dataset ka main data-quality kaam hai.

In [ ]:
with open("results.json", "w") as f:
    json.dump(RESULTS, f, indent=2, default=str)
print("Saved results.json and figures/ folder")